In [4]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))  # notebooks/ → repo root

In [5]:
import logging
import numpy as np
import pandas as pd

from financialtools.processor import (
    Downloader,
    FundamentalTraderAssistant,
    # SCORED_METRICS,
    # _EMPTY_RESULT_KEYS,
    # _empty_result,
)
from financialtools.exceptions import EvaluationError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)
print("Imports OK")

Imports OK


In [6]:
# Download AAPL financials â€” requires yfinance and an active internet connection.
# from_ticker() never raises; on failure it returns an empty Downloader and logs the error.
import re


TICKER = "BRE.MI"
# SECTOR = "technology"   # key in sec_sector_metric_weights (lowercase)


d      = Downloader.from_ticker(TICKER)
merged = d.get_merged_data()
# for sector replace space with dash and lowercase, e.g. "consumer-cyclical" or "real-estate"   
SECTOR = d.get_info_data()["sector"].str.lower().to_string(index=False)
SECTOR = re.sub(" ", "-", SECTOR)
SECTOR
# print(f"merged shape : {merged.shape}")   # (time_periods, financial_columns)

# if merged.empty:
#     raise RuntimeError(f"Download failed for {TICKER} â€” check logs/error.log")

# merged[["ticker", "time", "total_revenue", "gross_profit", "free_cash_flow"]].head()

'consumer-cyclical'

In [ ]:
info = d.get_info_data()
# rich.print(info["longBusinessSummary"].str.lower().to_string(index=False))
info["longName"].str.lower().to_string(index=False)

brembo n.v., together with its subsidiaries, de...

In [5]:
from financialtools.config import sec_sector_metric_weights

# Build the weights DataFrame from the canonical sector config â€” single source of truth.
# AAPL is in "technology" per sec_sector_metric_weights (yfinance sectorKey convention).
sector_weights_dict = sec_sector_metric_weights[SECTOR]
weights = pd.DataFrame({
    "sector" : SECTOR,
    "metrics": list(sector_weights_dict.keys()),
    "weights": list(sector_weights_dict.values()),
})

In [23]:
# sec_sector_metric_weights['consumer-cyclical']

In [6]:
fta = FundamentalTraderAssistant(data=merged, weights=weights)
result = fta.evaluate()

In [8]:
# info[
# ['beta',
#  'ticker',
# 'longName',
# 'longBusinessSummary',
# 'recommendationKey',
# 'shortName', 
# 'symbol',
# 'targetHighPrice',
# 'targetLowPrice',
# 'targetMeanPrice',
# 'targetMedianPrice',]]

In [7]:
metrics = result['metrics']
extended_metrics = result['extended_metrics']
eval_metrics = result['eval_metrics']
composite_scores = result['composite_scores']
rf = result['red_flags']

In [8]:
import rich
# rich.print(extended_metrics)

In [9]:
# from metrics add variable year taking year from time column   
metrics['time'] = pd.to_datetime(metrics['time']).dt.year
extended_metrics['time'] = pd.to_datetime(extended_metrics['time']).dt.year
eval_metrics['time'] = pd.to_datetime(eval_metrics['time']).dt.year
composite_scores['time'] = pd.to_datetime(composite_scores['time']).dt.year
rf['time'] = pd.to_datetime(rf['time']).dt.year


In [10]:
from financialtools.utils import dataframe_to_json
metrics, extended_metrics, eval_metrics, composite_scores, rf = [
    dataframe_to_json(df)
    for df in [metrics, extended_metrics, eval_metrics, composite_scores, rf]
]

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify key is set before making any API calls
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY not set — add it to your .env file")
print("API key loaded:", api_key[:8] + "...")

API key loaded: sk-proj-...


In [12]:
from langchain_core.output_parsers import PydanticOutputParser
# from langchain.output_parsers import OutputFixingParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [13]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class StockRegimeAssessment(BaseModel):
    ticker: str = Field(
        ..., description='The ticker of the stock under analysis'
    )
    regime: Literal["bull", "bear"] = Field(
        ..., description="The fundamental regime classification of the stock"
    )
    regime_rationale: str = Field(
        ..., description="Concise explanation justifying the regime classification based on the financial metrics, composite ratio and red flags"
    )
    metrics_movement: str = Field(
        ..., description=(
            "A summary description of how key financial metrics have moved across years, "
            "e.g., 'GrossMargin increased steadily, DebtToEquity rose sharply, FCFYield remained stable.'"
        )
    )
    non_aligned_findings: Optional[str] = Field(
        None,
        description=(
            "Observations or signals that are not aligned with the overall metric trends, "
            "such as contradictory indicators, anomalies."
        )
    )
    evaluation: Literal["overvalued", "undervalued", "fair"] = Field(
        ..., description="The valuation of the stock based on the Evaluation metrics"
    )
    evaluation_rationale: str = Field(
        ..., description="Concise explanation justifying the evaluation classification based on the financial metrics, composite ratio and red flags"
    )


In [14]:
from financialtools.prompts import (
    system_prompt_StockRegimeAssessment_extended,
)

In [15]:
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
# parser = PydanticOutputParser(pydantic_object=StockRegimeAssessment)
# parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
# format_instructions = parser.get_format_instructions()
llm = llm.with_structured_output(StockRegimeAssessment)
# system_prompt_filled = system_prompt_StockRegimeAssessment_extended.format(format_instructions=format_instructions)
system_prompt_filled = system_prompt_StockRegimeAssessment_extended

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human", "Metrics:\n{metrics}\nExtended Metrics:\n{extended_metrics}\nScores:\n{composite_scores}\nEvaluation Metrics:\n{eval_metrics}\nRedFlags:\n{red_flags}"),
])

chain = prompt | llm 
response = chain.invoke({
    "metrics": metrics,
    "extended_metrics": extended_metrics,
    "composite_scores": composite_scores,
    "eval_metrics": eval_metrics,
    "red_flags": rf,
})

2026-04-09 23:40:04,462 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [16]:
import rich
rich.print(response)

StockRegimeAssessment(
    ticker='BRE.MI',
    regime='bear',
    regime_rationale="The company's composite score has declined to below 3, indicating weakening fundamentals. Key
metrics such as FCF yield and free cash flow to debt have deteriorated sharply, with FCF to debt showing 
insufficient free cash flow to cover debt obligations in 2024. Profitability margins remain stable but are not 
improving, and leverage remains high. The red flag regarding inadequate free cash flow to cover debt further 
signals increased financial risk. Overall, the deteriorating cash flow metrics, declining composite score, and red 
flag suggest a weakening, bearish fundamental regime.",
    metrics_movement='GrossMargin increased slightly, DebtToEquity decreased marginally, but FCFYield and FCFtoDebt
declined sharply, with FCF to debt becoming insufficient to cover debt in 2024.',
    non_aligned_findings='Despite stable profitability margins, the significant drop in free cash flow metrics and 
the red flag indicating insufficient free cash flow to cover debt contradict the overall stability suggested by 
margins.',
    evaluation='overvalued',
    evaluation_rationale='Valuation metrics such as high P/FCF ratio and declining free cash flow, combined with 
weak cash flow coverage and a composite score below 3, indicate the stock is overvalued relative to its 
deteriorating fundamentals.'
)

In [ ]:
from financialtools.prompts import (
    system_prompt_liquidity,
)
from financialtools.pydantic_models import LiquidityAssessment

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
parser = PydanticOutputParser(pydantic_object=LiquidityAssessment)
parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
format_instructions = parser.get_format_instructions()
system_prompt_filled = system_prompt_liquidity.format(format_instructions=format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human", "Metrics:\n{metrics}\nExtended Metrics:\n{extended_metrics}\nScores:\n{composite_scores}\nEvaluation Metrics:\n{eval_metrics}\nRedFlags:\n{red_flags}"),
])

chain = prompt | llm | parser
response = chain.invoke({
    "metrics": metrics,
    "extended_metrics": extended_metrics,
    "composite_scores": composite_scores,
    "eval_metrics": eval_metrics,
    "red_flags": rf,
})

rich.print(response)

2026-03-20 13:26:32,203 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-20 13:26:36,419 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


LiquidityAssessment(
    rating='adequate',
    rationale="The company's liquidity ratios (current ratio ~1.11-1.15, quick ratio ~1.09-1.12) are stable and 
indicate sufficient short-term liquidity. The cash ratio remains low (~0.18-0.20), suggesting limited immediate 
cash reserves. The working capital ratio is modest (~0.084-0.13), reflecting reliance on receivables and inventory.
The cash conversion cycle is highly negative (~ -626 to -460 days), indicating efficient receivables and payables 
management. However, high DSO (~63-100 days) and low cash reserves pose potential risks if collection or payment 
timing worsens. Overall, liquidity is stable but warrants monitoring for receivables and cash levels.",
    working_capital_efficiency='The working capital efficiency chain shows a highly negative CCC (~ -626 to -460 
days), driven by rapid receivables turnover (DIO ~0.67-0.75 days) and high receivables turnover (~4.86-5.79). This 
indicates efficient working capital management, with receivables collected and payables paid promptly, supporting 
liquidity. The DSO remains relatively high (~63-100 days), which could be a concern if collection periods 
lengthen.',
    concerns='Low cash ratio and high DSO levels pose risks to liquidity if receivables collection slows or cash 
inflows diminish. The persistent negative CCC, while indicative of efficiency, also suggests dependence on tight 
working capital cycles, which could be strained under adverse conditions.'
)

In [21]:
from financialtools.prompts import (
    system_prompt_solvency,
)
from financialtools.pydantic_models import SolvencyAssessment

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
parser = PydanticOutputParser(pydantic_object=SolvencyAssessment)
parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
format_instructions = parser.get_format_instructions()
system_prompt_filled = system_prompt_solvency.format(format_instructions=format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human", "Metrics:\n{metrics}\nExtended Metrics:\n{extended_metrics}\nScores:\n{composite_scores}\nEvaluation Metrics:\n{eval_metrics}\nRedFlags:\n{red_flags}"),
])

chain = prompt | llm | parser
response = chain.invoke({
    "metrics": metrics,
    "extended_metrics": extended_metrics,
    "composite_scores": composite_scores,
    "eval_metrics": eval_metrics,
    "red_flags": rf,
})

rich.print(response)

2026-03-20 13:28:13,006 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-20 13:28:15,985 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


SolvencyAssessment(
    rating='adequate',
    rationale="MAIRE.MI exhibits manageable leverage with Debt-to-Equity ratios around 1.6–1.8 and low Debt Ratios 
(~13-17%), consistent with sector norms. Net Debt to EBITDA remains below 1 in recent years, indicating sufficient 
earnings to cover debt. Interest coverage ratios are robust (>12x), reflecting strong capacity to service interest.
Although Debt Growth increased by 15% in 2024, this does not currently threaten solvency given stable leverage and 
profitability. Liquidity ratios (~1.1) support short-term obligations, with no significant red flags. Overall, the 
company's debt profile is balanced and sustainable.",
    debt_trend='The company is experiencing a moderate increase in debt in 2024 (+15%), indicating a slight 
deleveraging trend but not alarming. This suggests a cautious approach to debt management, with ongoing monitoring 
recommended to ensure leverage remains within manageable levels.',
    concerns='Slight upward trend in debt levels in 2024; ongoing monitoring recommended. Working capital ratios 
are modest, indicating limited buffer for unexpected liquidity needs. No significant concerns regarding liquidity 
or debt servicing capacity at this time.'
)

In [22]:
from financialtools.prompts import (
    system_prompt_cash_flow,
)
from financialtools.pydantic_models import CashFlowAssessment

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
parser = PydanticOutputParser(pydantic_object=CashFlowAssessment)
parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
format_instructions = parser.get_format_instructions()
system_prompt_filled = system_prompt_cash_flow.format(format_instructions=format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human", "Metrics:\n{metrics}\nExtended Metrics:\n{extended_metrics}\nScores:\n{composite_scores}\nEvaluation Metrics:\n{eval_metrics}\nRedFlags:\n{red_flags}"),
])

chain = prompt | llm | parser
response = chain.invoke({
    "metrics": metrics,
    "extended_metrics": extended_metrics,
    "composite_scores": composite_scores,
    "eval_metrics": eval_metrics,
    "red_flags": rf,
})

rich.print(response)

2026-03-20 13:30:37,378 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-20 13:30:42,780 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


CashFlowAssessment(
    rating='adequate',
    rationale='The company demonstrates positive free cash flow with improving cash conversion ratios over the 
period, indicating efficient cash generation from operations. FCF to revenue and FCF yield are modest but 
increasing, and FCF to debt ratios improve, reflecting manageable leverage. The negative Capex ratio suggests 
maintenance-level reinvestment, supporting cash flow sustainability. However, rising DSO and declining inventory 
turnover indicate potential liquidity and operational efficiency concerns that should be monitored.',
    capital_allocation='The company maintains a negative Capex ratio, implying reinvestment at maintenance levels 
or asset disposals, which supports free cash flow. The increasing FCFGrowth indicates growing cash flow, but high 
receivables days and inventory turnover decline suggest that operational efficiency and asset utilization should be
carefully managed to ensure productive reinvestment.',
    concerns='DSO remains high (~94 days), and inventory turnover is declining, which could impact liquidity and 
operational efficiency if not addressed. Despite positive cash flow metrics, these operational indicators warrant 
ongoing monitoring.'
)

In [23]:
from financialtools.prompts import (
    system_prompt_efficiency,
)
from financialtools.pydantic_models import EfficiencyAssessment

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
parser = PydanticOutputParser(pydantic_object=EfficiencyAssessment)
parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
format_instructions = parser.get_format_instructions()
system_prompt_filled = system_prompt_efficiency.format(format_instructions=format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human", "Metrics:\n{metrics}\nExtended Metrics:\n{extended_metrics}\nScores:\n{composite_scores}\nEvaluation Metrics:\n{eval_metrics}\nRedFlags:\n{red_flags}"),
])

chain = prompt | llm | parser
response = chain.invoke({
    "metrics": metrics,
    "extended_metrics": extended_metrics,
    "composite_scores": composite_scores,
    "eval_metrics": eval_metrics,
    "red_flags": rf,
})

rich.print(response)

2026-03-20 13:32:01,063 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-20 13:32:05,481 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


EfficiencyAssessment(
    rating='adequate',
    rationale="The company's asset turnover has improved over the years, indicating better utilization of assets. 
Despite this, the very high DPO and DSO result in a persistently negative CCC, suggesting reliance on extended 
payables and slow receivables collection. These factors point to moderate operational efficiency with room for 
improvement, especially in receivables management.",
    working_capital_chain='Receivables Turnover and DSO show that collection speed remains slow, with DSO 
decreasing slightly over time but still high. Inventory Turnover and DIO indicate efficient inventory management, 
with DIO remaining low. Payables Turnover and DPO are very high, reflecting extended payment periods to suppliers. 
The negative CCC persists across all years, driven by high DPO and DSO, indicating reliance on supplier credit to 
finance operations. Multi-year trends show slight improvements in asset turnover and receivables collection, but 
the overall working capital cycle remains elongated, posing potential liquidity risks if supplier terms change.',
    concerns='The reliance on extended payables and slow receivables collection could pose liquidity risks if 
supplier terms tighten or if collection efficiency deteriorates further.'
)

In [26]:
from financialtools.prompts import (
    system_prompt_growth,
)
from financialtools.pydantic_models import GrowthAssessment

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
parser = PydanticOutputParser(pydantic_object=GrowthAssessment)
parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
format_instructions = parser.get_format_instructions()
system_prompt_filled = system_prompt_growth.format(format_instructions=format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human", "Metrics:\n{metrics}\nExtended Metrics:\n{extended_metrics}\nScores:\n{composite_scores}\nEvaluation Metrics:\n{eval_metrics}\nRedFlags:\n{red_flags}"),
])

chain = prompt | llm | parser
response = chain.invoke({
    "metrics": metrics,
    "extended_metrics": extended_metrics,
    "composite_scores": composite_scores,
    "eval_metrics": eval_metrics,
    "red_flags": rf,
})

rich.print(response)

2026-03-20 13:37:00,751 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-20 13:37:03,512 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


GrowthAssessment(
    trajectory='accelerating',
    rationale="The company's revenue, net income, and FCF metrics have shown consistent and significant 
year-over-year growth, with revenue increasing by approximately 38.5%, net income by 58.5%, and FCF by 33.3% in 
2024. The composite growth scores have also improved each year, indicating an accelerating growth trajectory. While
there are some concerns about increasing debt levels and FCF yield pressures, the overall growth signals and 
profitability improvements suggest the company is in an accelerating growth phase, supported by strong cash flow 
generation and efficiency gains.",
    dilution_impact='The analysis does not specify changes in shares outstanding; however, the positive growth in 
earnings and cash flows suggests that dilution effects are not offsetting growth significantly. If share buybacks 
are occurring, they could be offsetting dilution, but this is not explicitly stated.',
    concerns='Increasing debt levels (~15.3% growth) and negative FCF growth (-26.9%) in 2024 could pose future 
liquidity or leverage risks if trends persist. Monitoring debt management and cash flow sustainability is 
advisable.'
)

In [25]:
from financialtools.prompts import (
    system_prompt_profitability,
)
from financialtools.pydantic_models import ProfitabilityAssessment

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
parser = PydanticOutputParser(pydantic_object=ProfitabilityAssessment)
parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
format_instructions = parser.get_format_instructions()
system_prompt_filled = system_prompt_profitability.format(format_instructions=format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human", "Metrics:\n{metrics}\nExtended Metrics:\n{extended_metrics}\nScores:\n{composite_scores}\nEvaluation Metrics:\n{eval_metrics}\nRedFlags:\n{red_flags}"),
])

chain = prompt | llm | parser
response = chain.invoke({
    "metrics": metrics,
    "extended_metrics": extended_metrics,
    "composite_scores": composite_scores,
    "eval_metrics": eval_metrics,
    "red_flags": rf,
})

rich.print(response)

2026-03-20 13:34:02,489 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-20 13:34:05,848 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


ProfitabilityAssessment(
    rating='adequate',
    rationale='From 2021 to 2024, MAIRE.MI shows a clear improvement in profitability metrics. Initial margins were
thin, with gross margins around 65% and net profit margins near 3%, and returns such as ROE at 17% and ROA at 1.6% 
indicating weak profitability. Over the years, margins and returns have steadily increased, with 2024 margins 
reaching approximately 61-63%, ROE at 33%, and ROIC at 15%, reflecting stronger operational efficiency and capital 
utilization. The composite score has risen above 3, indicating a transition toward a strong profitability profile. 
Cash flow metrics, while still showing some concerns with negative accruals in earlier years, have improved, 
supporting the view of increasing earnings quality. Despite high leverage and some volatility in growth rates, the 
overall trend suggests a solid and improving profitability position.',
    earnings_quality='negative',
    concerns='High debt-to-equity ratio (~1.78), negative accruals in earlier years, and some volatility in free 
cash flow growth warrant monitoring, but overall profitability has strengthened significantly.'
)

In [27]:
from financialtools.prompts import (
    system_prompt_red_flags,
)
from financialtools.pydantic_models import RedFlagsAssessment

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
parser = PydanticOutputParser(pydantic_object=RedFlagsAssessment)
parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
format_instructions = parser.get_format_instructions()
system_prompt_filled = system_prompt_red_flags.format(format_instructions=format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human", "Metrics:\n{metrics}\nExtended Metrics:\n{extended_metrics}\nScores:\n{composite_scores}\nEvaluation Metrics:\n{eval_metrics}\nRedFlags:\n{red_flags}"),
])

chain = prompt | llm | parser
response = chain.invoke({
    "metrics": metrics,
    "extended_metrics": extended_metrics,
    "composite_scores": composite_scores,
    "eval_metrics": eval_metrics,
    "red_flags": rf,
})

rich.print(response)

2026-03-20 13:37:54,745 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


RedFlagsAssessment(
    severity='none',
    rationale='The company exhibits generally healthy profitability margins, manageable leverage levels, and 
positive cash flow metrics across the observed years. Debt-to-equity ratios remain within sector norms, and 
liquidity ratios are adequate. No trigger flags or concerning anomalies are present in the provided metrics.',
    cash_flow_flags=None,
    threshold_flags=None,
    quality_concerns=None
)

2026-03-20 13:39:00,495 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-20 13:39:07,057 [INFO] httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


OutputParserException: Failed to parse ComprehensiveStockAssessment from completion {"ticker": "MAIRE.MI", "regime": "bull", "regime_rationale": "The company's financial metrics show consistent improvement across profitability, cash flow, and operational efficiency, with manageable leverage and no red flags. Valuation metrics indicate undervaluation, supporting a bullish outlook.", "evaluation": "undervalued", "liquidity": {"rating": "adequate", "rationale": "Liquidity ratios such as current and quick ratios are above 1, indicating sufficient short-term liquidity. Cash ratio is low but typical for the sector, and working capital is positive, suggesting manageable liquidity position.", "working_capital_efficiency": "Working capital ratios are stable; DSO and DIO are high but decreasing over time, indicating improving receivables and inventory management. DPO remains stable, contributing to a stable cash conversion cycle.", "concerns": null}, "solvency": {"rating": "adequate", "rationale

In [1]:
from agents import create_financial_manager
agent = create_financial_manager(model="gpt-4.1-nano")
config = {"configurable": {"thread_id": "session-1"}}

result = agent.invoke(
      {"messages": [{"role": "user", "content": "Provide a quantitative snapshot of the financial health of BRE.MI for the chief trader. "
      "one section per topic analyzed with key insights and actionable recommendations. "
      "Conclude with an overall assessment of the stock's financial health and suitability for trading (long/short/neutral)."}]},  
      config=config,
  )
import rich
rich.print(result["messages"][-1].content)

BRE.MI
4
BRE.MI
4


The financial data for Brembo N.V. has been successfully prepared and cached under the key "BRE.MI_all." I will now
proceed with the analyses for each category—liquidity, solvency, profitability, efficiency, cash flow, growth, and 
red flags—and then compile a detailed, structured report with insights and actionable recommendations.
**Liquidity Analysis**  
Brembo N.V. exhibits a healthy liquidity profile with a current ratio above industry average, indicating sufficient
short-term assets to cover liabilities. The quick ratio is also robust, reflecting strong liquid assets, and the 
cash ratio confirms ample cash holdings. The working capital ratio is positive, supporting operational stability. 
Efficiency metrics, such as DSO, DIO, DPO, and CCC, suggest effective management of receivables, inventories, and 
payables, contributing to optimal cash conversion cycles.  
***Key Insight:*** The company maintains liquidity buffers that can sustain operational needs and short-term 
obligations.  
***Actionable Recommendation:*** Maintain focus on receivables and inventory management to preserve liquidity 
levels, especially during market fluctuations.  

**Solvency Analysis**  
Brembo N.V. displays moderate leverage levels, with debt-to-equity and debt ratios within acceptable ranges, 
indicating balanced use of debt financing. The equity ratio underscores a well-capitalized structure. Net debt to 
EBITDA is manageable, and interest coverage ratios reveal comfortable debt servicing capacity. The debt growth 
trend appears stable without rapid increases, signaling prudent long-term financial management.  
***Key Insight:*** The company has a solid long-term solvency position, reducing financial risk exposure.  
***Actionable Recommendation:*** Continue monitoring debt levels and interest coverage to preempt any potential 
rise in leverage, especially in fluctuations of interest rates or operational performance.

**Profitability Analysis**  
Brembo N.V. demonstrates strong profitability margins: high gross margin and operating margin indicate effective 
cost control and pricing strategies. Net profit margin remains healthy, with ROA, ROE, and ROIC metrics reflecting 
efficient capital utilization and shareholder value creation. The earnings quality, assessed via the accruals 
ratio, suggests stable and reliable earnings streams.  
***Key Insight:*** The company is consistently profitable, with margins and returns supporting trading 
attractiveness.  
***Actionable Recommendation:*** Leverage profitability strengths to fund growth initiatives or shareholder returns
while maintaining cost discipline.

**Efficiency Analysis**  
Asset turnover ratios are aligned with industry standards, showing efficient utilization of assets. Receivables and
inventory turnover metrics are solid, indicating swift collection and inventory management. Payables turnover 
aligns with supplier terms, supporting working capital efficiency. The DSO/DIO/DPO chain underscores effective 
operational cycle management, enhancing cash flow.  
***Key Insight:*** Operational efficiency mitigates liquidity strain and enhances overall financial health.  
***Actionable Recommendation:*** Continue optimizing inventory and receivables processes to sustain operational 
efficiency and reduce working capital requirements.

**Cash Flow Analysis**  
Free cash flow to revenue is positive, indicating effective cash generation from core activities. FCF yield and 
to-debt ratios underscore good cash flow relative to obligations. Cash conversion cycle management is effective, 
and capex-to-depreciation ratios suggest prudent investment. FCF growth trend remains favorable, supporting future 
flexibility.  
***Key Insight:*** Brembo N.V. consistently generates healthy cash flows, underpinning financial resilience.  
***Actionable Recommendation:*** Maintain disciplined capital expenditures to sustain cash flow quality and support
strategic growth.

**Growth Analysis**  
Revenue and net i

In [6]:
result = agent.invoke(
      {"messages": [{"role": "user", "content": "Provide a quantitative trading analysis of the Profitability for A2A.MI in the Utilities sector?"}]},  
      config=config,
  )
import rich
rich.print(result["messages"][-1].content)

The profitability analysis for A2A.MI in the Utilities sector indicates:

- **Profit Margins:**  
  - Gross Margin: 20.7%  
  - Operating Margin: 7.1%  
  - Net Profit Margin: 4.5%  
  These margins have improved, reflecting better operational efficiency.

- **Return Ratios:**  
  - Return on Equity (ROE): 15.5%  
  - Return on Assets (ROA): 3.5%  
  Both indicate higher efficiency in generating returns, improving the company's profitability outlook.

- **Earnings Quality:**  
  - High positive accruals suggest reported earnings may not be fully supported by cash flows and could be 
inflated, raising concerns about earnings sustainability.

- **Free Cash Flow:**  
  - Negative at -2.3%, with an FCF yield of -4.6%, indicating the company is not generating enough cash from 
operations to cover capital expenditures and debt obligations.  
  - With high net debt relative to EBITDA (~2.44), persistent negative free cash flow poses a liquidity risk.

**Summary:**  
While profitability margins and returns are improving, negative free cash flow and earnings quality issues 
highlight ongoing risks that could impact the company's ability to sustain profitability and manage debt 
effectively.

Would you like this as a concise executive summary for trading decisions?

In [ ]:
result = agent.invoke(
      {"messages": [{"role": "user", "content": "Is BRE.MI a good candidate for short selling?"}]},  
      config=config,
  )
import rich
rich.print(result["messages"][-1].content)

Based on the available quantitative and qualitative analysis:

- A2A.MI shows improved profit margins and higher return ratios, indicating operational efficiency and 
profitability growth.
- However, persistent negative free cash flow (-2.3%) and high earnings accruals suggest earnings may be inflated 
and cash generation weak.
- The company’s liquidity position remains fragile, with limited cash reserves and ongoing cash flow concerns.
- The sector’s monetary environment, competitive dynamics, and specific company risk factors are not fully detailed
here but could influence the decision.

**Conclusion:**  
While certain risk factors like weak cash flow and earnings quality issues could make A2A.MI a candidate for short 
at a point of notable deterioration, current profitability signals and sector trends do not strongly support a 
clear bearish case. Additional sector-wide and macroeconomic context, along with forward-looking indicators, are 
needed for a robust short selling decision.

**Recommendation:**  
Further analysis of recent cash flow trends, sector performance, and technical signals is advisable before 
considering short selling. The current data suggests caution rather than an outright short candidate.

In [12]:
from agents import create_financial_manager

agent  = create_financial_manager()
config = {"configurable": {"thread_id": "session-1"}}

# Blocking invoke
result = agent.invoke(
    {"ticker": "IP.MI"},
    config=config,
)
print(result["final_report"])

## Position Recommendation

**NO POSITION — Medium conviction**

Interpump’s financial profile is stable, with adequate liquidity and solvency metrics, and moderate profitability. However, the decelerating growth trajectory and extended working capital cycles (DIO of 184 days, CCC of 176 days) raise concerns about operational momentum and cash conversion efficiency. The company’s cash flow profile, while improving, remains modest relative to market cap, and the persistent negative Capex ratio suggests potential long-term asset renewal risks. Given these factors, I see no compelling near-term catalyst to justify a long or short stance, but the deteriorating growth trend warrants caution.

The primary risk to this view is a sudden improvement in growth or margin expansion driven by operational efficiencies or market share gains, which could trigger a reassessment of the long case. Conversely, a further slowdown or deterioration in working capital management (e.g., DSO or DIO worsening be

In [13]:
import rich
rich.print(result["final_report"])

## Position Recommendation

**NO POSITION — Medium conviction**

Interpump’s financial profile is stable, with adequate liquidity and solvency metrics, and moderate profitability. 
However, the decelerating growth trajectory and extended working capital cycles (DIO of 184 days, CCC of 176 days) 
raise concerns about operational momentum and cash conversion efficiency. The company’s cash flow profile, while 
improving, remains modest relative to market cap, and the persistent negative Capex ratio suggests potential 
long-term asset renewal risks. Given these factors, I see no compelling near-term catalyst to justify a long or 
short stance, but the deteriorating growth trend warrants caution.

The primary risk to this view is a sudden improvement in growth or margin expansion driven by operational 
efficiencies or market share gains, which could trigger a reassessment of the long case. Conversely, a further 
slowdown or deterioration in working capital management (e.g., DSO or DIO worsening beyond current levels) could 
accelerate downside. I recommend a medium-term horizon (3–12 months) to monitor these dynamics, with a focus on 
revenue growth stabilization and cash flow sustainability.

---

## Scorecard

| Topic             | Rating / Trajectory | Signal | Long Catalyst | Short Catalyst |
|-------------------|---------------------|---------|----------------|----------------|
| Liquidity         | Adequate / Stable  | ✅      | Stable liquidity ratios support resilience | Deterioration in 
quick or cash ratios below 1.0 |
| Solvency          | Adequate / Improving | ✅     | Continued debt reduction and interest coverage >8 | Debt 
ratios rising above sector thresholds or interest coverage falling below 5x |
| Profitability     | Adequate / Stable  | ✅      | Margins stabilizing or expanding | Margins compressing below 
10% net margin |
| Efficiency        | Adequate / Deteriorating | ⚠️  | Improvement in DSO/DIO reducing CCC below 150 days | DSO or 
DIO extending further, worsening liquidity |
| Cash Flow         | Adequate / Improving | ⚠️  | FCF yield stabilizes above 5.5% | FCF turns negative or drops 
below 3% yield |
| Growth            | Decelerating / Negative | ⚠️  | Revenue stabilizes or resumes growth >2% | Continued decline 
in revenue or net income |
| Red Flags         | None / Stable      | ⚠️      | No red flags emerging | Red flags such as rising debt or 
declining margins |

---

## Deep-Dive by Topic

### 1. Liquidity
**Verdict:** Adequate / Stable — supports neither long nor short bias.  
**Key metrics:**  
- Current ratio: ~1.90  
- Quick ratio: ~1.07  
- Cash ratio: ~0.44  
- Working capital ratio: ~0.47  
- Composite score: 3.38  
**Trend:** Stable, with steady improvement in working capital efficiency →  
**Insight:** The company maintains a comfortable liquidity buffer, but the modest cash ratio (~0.44) indicates 
limited cushion for sudden liquidity shocks.  
**Trigger:** Quick ratio falling below 1.0 or cash ratio below 0.4 would signal liquidity stress.

### 2. Solvency
**Verdict:** Adequate / Improving — no immediate concern.  
**Key metrics:**  
- Debt-to-Equity: 0.399 (2024) vs 0.579 (2022)  
- Debt ratio: 0.236 (2024) vs 0.287 (2022)  
- Net Debt/EBITDA: 0.71 (2024) vs 0.97 (2022)  
- Interest coverage: ~8.2x (2024) vs >37x (2022)  
**Trend:** Leverage decreasing, debt metrics improving →  
**Insight:** The company is actively deleveraging, which enhances financial stability, though interest coverage has
declined from very high levels.  
**Trigger:** Debt-to-Equity exceeding 0.6 or interest coverage dropping below 5x.

### 3. Profitability
**Verdict:** Adequate / Stable — supports neutral stance.  
**Key metrics:**  
- Gross margin: ~34-35%  
- Net margin: 10-13%  
- ROA: 6-8%  
- ROE: 11-17%  
- ROIC: 9.5-11.7%  
- FCF yield: 4-5.5%  
**Trend:** Margins stable, earnings increasingly cash-backed →  
**Insight:** The company’s profitability is consistent, with cash flow quality improving, but no 

In [8]:
from financialtools.analysis import (
    _build_weights,
    _filter_year,
    _normalise_time,
)

from financialtools.processor import Downloader, FundamentalTraderAssistant

import re

In [ ]:

d = Downloader.from_ticker('BGN.MI')
merged = d.get_merged_data()
info_df = d.get_info_data()

In [9]:
company_name = info_df["longName"].str.lower().to_string(index=False).strip()
raw = info_df["sector"].str.lower().to_string(index=False)
sector = re.sub(r" ", "-", raw.strip())
weights = _build_weights(sector)